## General Strategy

1. **Compress light curves into 3 SALT2 numbers**  
   Each SN is reduced to $(x_0, x_1, c)$.  
   These are the parameters in my CSV file.

2. **Define a distance estimator (Tripp relation)**  
   $
   \mu_{\text{obs}} = -2.5\log_{10}x_0 + \alpha x_1 - \beta c + M_B
   $
   where $\alpha, \beta, M_B$ are unknown *global nuisance parameters*.

3. **Get the theoretical expectation from cosmology**  
   For each redshift $z$, compute  
   $
   \mu_{\text{th}}(z; \Omega_M,H_0) =
   5\log_{10}\!\left(\frac{d_L(z)}{10\ \mathrm{pc}}\right),
   $  
   where $d_L(z)$ is the luminosity distance in your chosen cosmology  
   (e.g. flat $\Lambda$CDM).  
   In practice, $M_B$ absorbs $H_0$, so $H_0$ is fixed to a convenient value. So that's all you need. These things are analytically marginalizable. 

4. **Compare observed vs. theory**  
   Define the residuals:  
   $
   \Delta\mu_i = \mu_{\text{obs},i} - \mu_{\text{th}}(z_i).
   $

5. **Fit the nuisance parameters $\alpha,\beta,M_B$**  
   Choose them to minimize scatter in the Hubble diagram, i.e. minimize  
   $
   \chi^2 = \sum_i \frac{\Delta\mu_i^2}{\sigma_i^2},
   $  
   where $\sigma_i^2$ is the per-SN variance (measurement error + peculiar velocity + intrinsic scatter).  
   This is exactly what the iterative weighted least squares + MLE update for 
   $\sigma_{\rm int}$ is doing in the code.


   Reminder that M_b H_o are analytically marignailizable just like the project we did before. 

7. **Visualize**  
   - Plot $\mu_{\text{obs}}$ vs. $z$ (the Hubble diagram), overlay $\mu_{\text{th}}(z)$.  
   - Plot residuals vs. $z$, $x_1$, and $c$.  
   - Plot a histogram of residuals to show the scatter.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from astropy.cosmology import FlatLambdaCDM

# =========================
# CONFIG
# =========================
CSV = "ZTF_DESI_data/ZTF_snia_with_hosts_AND_SALT2.csv"
H0, OM0 = 70.0, 0.3          # cosmology for mu_th (MB absorbs H0 offset)
SIGMA_V = 300.0              # km/s peculiar velocity scatter
CUTS = dict(x1=(-3, 3), c=(-0.2, 0.8), fitprob_min=1e-7)

LOG10E = 1/np.log(10)

def mu0_from_x0(x0):
    """-2.5 log10 x0"""
    return -2.5*np.log10(x0)

def fit_alpha_beta_MB(df):
    # -------- Basic quality cuts --------
    mask = (
        np.isfinite(df["x0"]) & (df["x0"] > 0) &
        np.isfinite(df["x1"]) & np.isfinite(df["c"]) &
        np.isfinite(df["x0_err"]) & np.isfinite(df["x1_err"]) & np.isfinite(df["c_err"]) &
        np.isfinite(df["redshift"]) & (df["redshift"] > 0)
    )
    if "fitprob" in df.columns:
        mask &= (df["fitprob"] >= CUTS["fitprob_min"])
    mask &= df["x1"].between(*CUTS["x1"]) & df["c"].between(*CUTS["c"])
    d = df.loc[mask].copy()
    if len(d) < 10:
        raise ValueError(f"Too few SNe after cuts: {len(d)}")

    # -------- Columns --------
    x0, x1, c = d["x0"].values, d["x1"].values, d["c"].values
    x0e, x1e, ce = d["x0_err"].values, d["x1_err"].values, d["c_err"].values
    z = d["redshift"].values
    ze = d.get("redshift_err", pd.Series(np.zeros(len(d)), index=d.index)).values

    # Covariances (fill with zeros if missing)
    cov_x0_x1 = d.get("cov_x0_x1", pd.Series(np.zeros(len(d)), index=d.index)).values
    cov_x0_c  = d.get("cov_x0_c",  pd.Series(np.zeros(len(d)), index=d.index)).values
    cov_x1_c  = d.get("cov_x1_c",  pd.Series(np.zeros(len(d)), index=d.index)).values

    # -------- Cosmology & targets --------
    cosmo = FlatLambdaCDM(H0=H0, Om0=OM0)
    mu_th = cosmo.distmod(z).value
    mu0 = mu0_from_x0(x0)
    t = mu_th - mu0                      # target for linear part
    X = np.column_stack([x1, -c, np.ones_like(x1)])  # columns: [x1, -c, 1], shape (N,3)

    # -------- Variance helpers --------
    def sigma_meas_sq(alpha, beta):
        dmu_dx0 = -2.5*LOG10E*(1.0/x0)
        dmu_dx1 = alpha
        dmu_dc  = -beta
        var = (
            (dmu_dx0**2)*x0e**2 +
            (dmu_dx1**2)*x1e**2 +
            (dmu_dc**2)*ce**2 +
            2*dmu_dx0*dmu_dx1*cov_x0_x1 +
            2*dmu_dx0*dmu_dc *cov_x0_c  +
            2*dmu_dx1*dmu_dc *cov_x1_c
        )
        return np.clip(var, 0, None)

    def sigma_pv_sq():
        c_kms = 299_792.458
        term = (5/np.log(10))*(SIGMA_V/(c_kms*np.clip(z, 1e-4, None)))
        return term**2

    def sigma_z_sq():
        dz = np.maximum(ze, 1e-5)
        mu_p = FlatLambdaCDM(H0=H0, Om0=OM0).distmod(z+dz).value
        mu_m = FlatLambdaCDM(H0=H0, Om0=OM0).distmod(np.clip(z-dz, 1e-6, None)).value
        dmu_dz = (mu_p - mu_m)/(2*dz)
        return (dmu_dz*ze)**2

    # -------- Iterative WLS + sigma_int MLE --------
    alpha, beta, MB = 0.15, 3.0, -19.3   # initial guess
    sigma_int = 0.1
    for _ in range(6):
        v_meas = sigma_meas_sq(alpha, beta)
        v_tot = v_meas + sigma_pv_sq() + sigma_z_sq() + sigma_int**2
        w = 1.0/np.clip(v_tot, 1e-12, None)

        # Weighted least squares: ensure RHS is 1-D
        W = np.sqrt(w)[:, None]         # (N,1)
        Xw = W * X                      # (N,3)
        tw = (W[:, 0] * t)              # (N,)  <-- 1-D RHS
        theta, *_ = np.linalg.lstsq(Xw, tw, rcond=None)
        theta = np.asarray(theta).ravel()    # (3,)
        alpha, beta, MB = theta.tolist()     # scalars

        # MLE for sigma_int
        def nll(log_s):
            s = np.exp(log_s)
            v = v_meas + sigma_pv_sq() + sigma_z_sq() + s**2
            r = t - (alpha*x1 - beta*c + MB)
            return 0.5*np.sum(np.log(2*np.pi*np.clip(v,1e-12,None)) + (r*r)/np.clip(v,1e-12,None))
        res = minimize(nll, x0=np.log(max(sigma_int,1e-4)),
                       bounds=[(np.log(1e-4), np.log(0.5))])
        sigma_int = float(np.exp(res.x[0]))

    # -------- Parameter covariance from final WLS --------
    v_meas = sigma_meas_sq(alpha, beta)
    v_tot = v_meas + sigma_pv_sq() + sigma_z_sq() + sigma_int**2
    w = 1.0/np.clip(v_tot, 1e-12, None)
    XtWX = (X.T * w) @ X
    cov = np.linalg.inv(XtWX)
    se_alpha, se_beta, se_MB = np.sqrt(np.diag(cov))

    mu_obs = mu0 + alpha*x1 - beta*c + MB
    resid = mu_obs - mu_th
    chi2 = np.sum(w*resid*resid)
    ndof = len(resid) - 3

    out = dict(
        alpha=float(alpha), alpha_err=float(se_alpha),
        beta=float(beta),   beta_err=float(se_beta),
        MB=float(MB),       MB_err=float(se_MB),
        sigma_int=float(sigma_int),
        chi2_dof=float(chi2/max(ndof,1)),
        z=z, mu_obs=mu_obs, mu_th=mu_th,
        resid=resid, x1=x1, c=c
    )
    return out

# =========================
# RUN
# =========================
df = pd.read_csv(CSV)
res = fit_alpha_beta_MB(df)

print("Fitted (global scalars):")
print(f"  alpha = {res['alpha']:.3f} ± {res['alpha_err']:.3f}")
print(f"  beta  = {res['beta']:.3f} ± {res['beta_err']:.3f}")
print(f"  M_B   = {res['MB']:.3f} ± {res['MB_err']:.3f}")
print(f"  sigma_int = {res['sigma_int']:.3f}  |  chi2/dof = {res['chi2_dof']:.2f}")

# =========================
# PLOTS
# =========================
z, mu_obs, mu_th = res["z"], res["mu_obs"], res["mu_th"]
resid, x1, c = res["resid"], res["x1"], res["c"]

# 1) Hubble diagram
plt.figure(figsize=(7,5))
plt.scatter(z, mu_obs, s=8, label="SNe")
idx = np.argsort(z)
plt.plot(z[idx], mu_th[idx], label="ΛCDM", linewidth=2)
plt.xlabel("Redshift z")
plt.ylabel("Distance modulus μ")
plt.title("Hubble Diagram: μ_obs vs z")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# 2) Residuals vs z
plt.figure(figsize=(7,5))
plt.scatter(z, resid, s=8)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Redshift z")
plt.ylabel("Hubble residual (μ_obs − μ_th) [mag]")
plt.title("Residuals vs z")
plt.grid(True, alpha=0.3)
plt.show()

# 3) Residuals histogram
plt.figure(figsize=(7,5))
plt.hist(resid, bins=30)
plt.xlabel("Hubble residual [mag]")
plt.ylabel("Count")
plt.title("Residuals distribution")
plt.grid(True, alpha=0.3)
plt.show()

# 4a) Residuals vs x1
plt.figure(figsize=(7,5))
plt.scatter(x1, resid, s=8)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("x1")
plt.ylabel("Hubble residual [mag]")
plt.title("Residuals vs x1")
plt.grid(True, alpha=0.3)
plt.show()

# 4b) Residuals vs c
plt.figure(figsize=(7,5))
plt.scatter(c, resid, s=8)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("c")
plt.ylabel("Hubble residual [mag]")
plt.title("Residuals vs c")
plt.grid(True, alpha=0.3)
plt.show()